# 02 — Tire Lateral Force & Slip Angle

> **Engineering question:** How can a tire generate lateral force when the wheel is not pointing in exactly the same direction that the contact patch is traveling?

This notebook is the **live computational supplement** to the Lesson 02 slides and whiteboard work. The lesson still derives the geometry and local linearization by hand first; the notebook then pulls the **reviewed WUFR tire model itself** into the room so the figures are not screenshots or hand-drawn approximations.

**Teaching rhythm:** predict → derive → show the real model → move the operating point → interpret.

The source tire is the Hoosier 43105 R25B data set used as a reviewed engineering proxy for the intended WUFR R20 tire. It is not installed-car or track-correlation authority.

## 1. Load the reviewed tire runtime

The notebook does not carry copied tire curves. It loads the same exact processed TTC source exchange and reviewed adapter used by `pssd_tire`.

For this lesson we use one exact source state:

- normal load: **667 N**
- inclination: **0°**
- pressure: **82.7 kPa gauge** in the complete runtime source
- longitudinal slip: **0**
- source speed: approximately **40.2 km/h**

The older tire-selection summary rounds the pressure to 83 kPa, so its reported cornering stiffness is a useful cross-check rather than a byte-for-byte duplicate of the runtime state.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

from IPython.display import Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not find repository root from the current working directory")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pssd_tire import (
    R25B_CANONICAL_SOURCE_CONVENTION_ID,
    SteadyStateLateralOperatingState,
    evaluate_r25b_steady_state_lateral,
    load_r25b_steady_state_lateral_table,
)
from pssd_tire.r25b_named_runtime import (
    invert_r25b_classified_lateral_force,
    load_r25b_classified_lateral_table,
)


table = load_r25b_steady_state_lateral_table()

TARGET_FZ_N = 667.0
TARGET_GAMMA_RAD = math.radians(0.0)
TARGET_PRESSURE_PA = 82_700.0

matches = [
    curve
    for curve in table.curves
    if math.isclose(curve.normal_load_N, TARGET_FZ_N, abs_tol=1e-12)
    and math.isclose(curve.inclination_rad, TARGET_GAMMA_RAD, abs_tol=1e-12)
    and math.isclose(curve.pressure_Pa, TARGET_PRESSURE_PA, abs_tol=1e-9)
]
assert len(matches) == 1, f"Expected one exact runtime curve, found {len(matches)}"
curve = matches[0]

print(f"Runtime table:   {table.table_id}")
print(f"Curve:           {curve.curve_id}")
print(f"Source tire:     {curve.source_tire_id}")
print(f"Intended tire:   {curve.intended_tire_id}")
print(f"Adapter:         {curve.adapter_id}")
print(f"Convention:      {curve.source_convention_id}")
print(f"Fidelity:        {curve.fidelity_label}")

## 2. Sign convention before any plot

WUFR's canonical tire-contact frame is right-handed:

- (+x_t): forward in the wheel plane projected onto the road
- (+z_t): road-normal upward
- (+y_t = +z_t \times +x_t): leftward
- force role: **road on tire**

For positive forward transport,

[
\alpha = -\operatorname{atan2}(v_{y,t},v_{x,t})
]

and in the vehicle plane,

[
\boxed{\alpha = \delta - \hat\beta}
]

where (delta) is wheel heading and (hat\beta) is wheel-center velocity heading, both positive left.

The reviewed R25B runtime adapter transforms the historical/source SAE force sign into this canonical left-positive frame. Therefore the local (F_y)-versus-(alpha) slope through zero is positive.

## 3. First visual — wheel heading is not path direction

Use this immediately after the whiteboard derivation. The point is geometric, not yet a tire-force calculation.

In [ ]:
delta_deg = 6.0
beta_deg = 3.5
alpha_deg = delta_deg - beta_deg

delta = math.radians(delta_deg)
beta = math.radians(beta_deg)

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0.0, linewidth=1)
ax.axvline(0.0, linewidth=1)

# Vehicle +x reference
ax.arrow(0, 0, 1.15, 0, width=0.008, head_width=0.07, length_includes_head=True)
ax.text(1.18, 0, "vehicle +x", va="center")

# Wheel heading
ax.arrow(
    0,
    0,
    math.cos(delta),
    math.sin(delta),
    width=0.012,
    head_width=0.08,
    length_includes_head=True,
)
ax.text(
    0.82 * math.cos(delta),
    0.82 * math.sin(delta) + 0.08,
    rf"wheel heading $\delta={delta_deg:.1f}^\circ$",
    ha="center",
)

# Wheel-center velocity direction
ax.arrow(
    0,
    0,
    math.cos(beta),
    math.sin(beta),
    width=0.012,
    head_width=0.08,
    length_includes_head=True,
)
ax.text(
    0.80 * math.cos(beta),
    0.80 * math.sin(beta) - 0.11,
    rf"velocity heading $\hat{{\beta}}={beta_deg:.1f}^\circ$",
    ha="center",
)

ax.text(
    0.04,
    0.26,
    rf"$\alpha=\delta-\hat{{\beta}}={alpha_deg:.1f}^\circ$",
    fontsize=14,
)

ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-0.08, 1.30)
ax.set_ylim(-0.28, 0.50)
ax.set_axis_off()
ax.set_title("Slip angle is the difference between wheel heading and wheel-center motion")
plt.show()

## 4. Pull the real (F_y(\alpha)) curve from `pssd_tire`

This is the figure to put up over the slides once the room has predicted its shape.

Nothing below fits a new tire law. The samples are the reviewed runtime curve after the source-to-canonical adapter.

In [ ]:
alpha_deg_samples = [math.degrees(value) for value in curve.slip_angle_rad]
fy_N_samples = list(curve.lateral_force_N)

fig, ax = plt.subplots(figsize=(10.5, 6.2))
ax.plot(alpha_deg_samples, fy_N_samples, linewidth=2.5, label="reviewed R25B runtime curve")
ax.axhline(0.0, linewidth=1)
ax.axvline(0.0, linewidth=1)

ax.set_xlabel(r"Slip angle, $\alpha$ [deg]")
ax.set_ylabel(r"Lateral force, $F_y$ [N]")
ax.set_title(
    r"Reviewed WUFR tire model: $F_y(\alpha)$"
    "\n"
    r"$F_z=667$ N, $\gamma=0^\circ$, $P=82.7$ kPa gauge"
)
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

print(f"Supported slip domain: {min(alpha_deg_samples):.1f}° to {max(alpha_deg_samples):.1f}°")
print(f"Samples in this curve: {len(alpha_deg_samples)}")

### What to ask while this plot is on screen

Before touching Python again, have the room identify:

1. the near-linear region;
2. where the incremental slope starts declining;
3. the peak region;
4. whether the source shows post-peak behavior on each signed side;
5. why one (F_y) demand could correspond to multiple slip angles.

The curve should answer the question visually before the code answers it numerically.

## 5. Local linearization — compare the tangent with the tire

The general small-slip linearization is

[
F_y(\alpha) \approx F_y(0) + C_\alpha\alpha
]

with

[
C_\alpha = \left.\frac{\partial F_y}{\partial\alpha}\right|_{\alpha=0}.
]

Notice that we do **not** force (F_y(0)=0). The real source data may carry a small zero-slip force offset.

In [ ]:
zero_state = SteadyStateLateralOperatingState(
    slip_angle_rad=0.0,
    normal_load_N=TARGET_FZ_N,
    inclination_rad=TARGET_GAMMA_RAD,
    pressure_Pa=TARGET_PRESSURE_PA,
    state_id="LESSON02_ZERO_SLIP",
    source_id=curve.source_tire_id,
    source_convention_id=R25B_CANONICAL_SOURCE_CONVENTION_ID,
)
zero_response = evaluate_r25b_steady_state_lateral(zero_state, table=table)

fy0_N = zero_response.lateral_force_N
left_slope = zero_response.left_segment_slope_N_per_rad
right_slope = zero_response.right_segment_slope_N_per_rad

if zero_response.derivative_unique:
    c_alpha_N_per_rad = left_slope
else:
    c_alpha_N_per_rad = 0.5 * (left_slope + right_slope)

c_alpha_N_per_deg = c_alpha_N_per_rad * math.pi / 180.0

rows = [
    ("Runtime $F_y(0)$", fy0_N, "N"),
    ("Left local slope", left_slope, "N/rad"),
    ("Right local slope", right_slope, "N/rad"),
    ("Teaching $C_\\alpha$", c_alpha_N_per_rad, "N/rad"),
    ("Teaching $C_\\alpha$", c_alpha_N_per_deg, "N/deg"),
]
table_md = [
    "| Quantity | Value | Unit |",
    "|---|---:|---|",
    *[f"| {name} | {value:.3f} | {unit} |" for name, value, unit in rows],
]
display(Markdown("\n".join(table_md)))

print("Derivative unique at alpha=0:", zero_response.derivative_unique)
print("Rounded historical summary cross-check: about 523 N/deg at 83 kPa.")

In [ ]:
# Draw the tangent only over a small-slip teaching window.
linear_window_deg = [
    value / 20.0
    for value in range(-60, 61)  # -3° to +3°
]
linear_window_rad = [math.radians(value) for value in linear_window_deg]
tangent_fy_N = [
    fy0_N + c_alpha_N_per_rad * alpha
    for alpha in linear_window_rad
]

fig, ax = plt.subplots(figsize=(10.5, 6.2))
ax.plot(alpha_deg_samples, fy_N_samples, linewidth=2.5, label="reviewed runtime curve")
ax.plot(
    linear_window_deg,
    tangent_fy_N,
    linestyle="--",
    linewidth=2.0,
    label=r"local tangent $F_y(0)+C_\alpha\alpha$",
)
ax.scatter([0.0], [fy0_N], zorder=5, label=r"$F_y(0)$")
ax.axhline(0.0, linewidth=1)
ax.axvline(0.0, linewidth=1)

ax.set_xlabel(r"Slip angle, $\alpha$ [deg]")
ax.set_ylabel(r"Lateral force, $F_y$ [N]")
ax.set_title("The cornering-stiffness model is a tangent, not the whole tire")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## 6. The 800 N teaching demand — hand estimate versus real inverse

The slide/whiteboard example uses the rounded historical summary value

[
C_\alpha \approx 523\;\mathrm{N/deg}
]

and, for a deliberately simplified hand estimate, assumes (F_y(0)\approx0):

[
|\alpha| \approx \frac{800}{523} \approx 1.53^\circ.
]

Now compare that estimate with the real runtime curve. Because the actual source is signed and nonmonotonic, use the reviewed **named positive pre-peak branch policy** rather than guessing which root to keep.

In [ ]:
TEACHING_DEMAND_N = 800.0
SUMMARY_C_ALPHA_N_PER_DEG = 523.0

alpha_hand_deg = TEACHING_DEMAND_N / SUMMARY_C_ALPHA_N_PER_DEG
alpha_runtime_linear_deg = math.degrees(
    (TEACHING_DEMAND_N - fy0_N) / c_alpha_N_per_rad
)

classified_table = load_r25b_classified_lateral_table()
inverse = invert_r25b_classified_lateral_force(
    normal_load_N=TARGET_FZ_N,
    inclination_rad=TARGET_GAMMA_RAD,
    pressure_Pa=TARGET_PRESSURE_PA,
    requested_lateral_force_N=TEACHING_DEMAND_N,
    state_id="LESSON02_800N_DEMAND",
    branch_selector="positive_slip_pre_peak",
    table=classified_table,
)

assert inverse.selected_candidate is not None
alpha_model_deg = math.degrees(inverse.selected_candidate.slip_angle_rad)

comparison = [
    ("Hand estimate, rounded summary", alpha_hand_deg),
    ("Runtime local tangent incl. offset", alpha_runtime_linear_deg),
    ("Reviewed nonlinear pre-peak inverse", alpha_model_deg),
]
table_md = [
    "| Method | Slip angle |",
    "|---|---:|",
    *[f"| {name} | {value:.3f}° |" for name, value in comparison],
]
display(Markdown("\n".join(table_md)))

print(f"Inverse status: {inverse.status}")
print(f"Selected branch: {inverse.selected_candidate.branch_id}")

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 6.2))
ax.plot(alpha_deg_samples, fy_N_samples, linewidth=2.5, label="reviewed runtime curve")
ax.plot(
    linear_window_deg,
    tangent_fy_N,
    linestyle="--",
    linewidth=2.0,
    label="local tangent",
)
ax.axhline(TEACHING_DEMAND_N, linestyle=":", linewidth=1.5, label="800 N teaching demand")
ax.scatter(
    [alpha_model_deg],
    [TEACHING_DEMAND_N],
    s=80,
    zorder=5,
    label=rf"nonlinear pre-peak solution: $\alpha={alpha_model_deg:.2f}^\circ$",
)

ax.set_xlabel(r"Slip angle, $\alpha$ [deg]")
ax.set_ylabel(r"Lateral force, $F_y$ [N]")
ax.set_title("Same force demand, different model fidelity")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## 7. Live operating-point explorer

This is the **one widget worth using in Lesson 02**.

Do not lead with it. First have the room predict what moving the slider should do. Then use it as the Roman-style visual supplement: the point moves on the reviewed curve while the readout compares the real nonlinear response with the local tangent.

Things to call out live:

- near zero, model and tangent nearly agree;
- moving farther out, the residual grows;
- near the peak, (dF_y/d\alpha) is much smaller than (C_\alpha);
- post-peak, more slip angle does not guarantee more lateral force.

In [ ]:
alpha_min_deg = math.degrees(curve.slip_angle_rad[0])
alpha_max_deg = math.degrees(curve.slip_angle_rad[-1])

alpha_slider = widgets.FloatSlider(
    value=0.0,
    min=alpha_min_deg,
    max=alpha_max_deg,
    step=0.25,
    description="α [deg]",
    continuous_update=True,
    readout_format=".2f",
    layout=widgets.Layout(width="70%"),
    style={"description_width": "70px"},
)


def show_operating_point(alpha_deg: float) -> None:
    alpha_rad = math.radians(alpha_deg)
    state = SteadyStateLateralOperatingState(
        slip_angle_rad=alpha_rad,
        normal_load_N=TARGET_FZ_N,
        inclination_rad=TARGET_GAMMA_RAD,
        pressure_Pa=TARGET_PRESSURE_PA,
        state_id="LESSON02_LIVE_POINT",
        source_id=curve.source_tire_id,
        source_convention_id=R25B_CANONICAL_SOURCE_CONVENTION_ID,
    )
    response = evaluate_r25b_steady_state_lateral(state, table=table)

    tangent_N = fy0_N + c_alpha_N_per_rad * alpha_rad
    residual_N = response.lateral_force_N - tangent_N
    local_slope = 0.5 * (
        response.left_segment_slope_N_per_rad
        + response.right_segment_slope_N_per_rad
    )
    local_slope_N_per_deg = local_slope * math.pi / 180.0

    fig, ax = plt.subplots(figsize=(10.5, 6.2))
    ax.plot(alpha_deg_samples, fy_N_samples, linewidth=2.5, label="reviewed runtime curve")
    ax.plot(
        linear_window_deg,
        tangent_fy_N,
        linestyle="--",
        linewidth=2.0,
        label="zero-slip tangent",
    )
    ax.scatter([alpha_deg], [response.lateral_force_N], s=100, zorder=6, label="current point")
    ax.axhline(0.0, linewidth=1)
    ax.axvline(0.0, linewidth=1)

    ax.set_xlabel(r"Slip angle, $\alpha$ [deg]")
    ax.set_ylabel(r"Lateral force, $F_y$ [N]")
    ax.set_title(
        rf"$\alpha={alpha_deg:.2f}^\circ$   |   "
        rf"$F_y={response.lateral_force_N:.0f}$ N   |   "
        rf"local slope $\approx {local_slope_N_per_deg:.0f}$ N/deg"
    )
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    plt.show()

    display(
        Markdown(
            f"""
**Current operating point**

- nonlinear runtime: **{response.lateral_force_N:.1f} N**
- zero-slip tangent: **{tangent_N:.1f} N**
- nonlinear minus tangent: **{residual_N:+.1f} N**
- participating source curve: `{response.curve_id}`
- extrapolation: **none**
"""
        )
    )


interactive = widgets.interactive_output(
    show_operating_point,
    {"alpha_deg": alpha_slider},
)
display(alpha_slider, interactive)

## 8. Sanity checks

A plotted curve is still an engineering result. Check the model boundary explicitly.

In [ ]:
assert curve.normal_load_N == TARGET_FZ_N
assert math.isclose(curve.inclination_rad, TARGET_GAMMA_RAD, abs_tol=1e-12)
assert math.isclose(curve.pressure_Pa, TARGET_PRESSURE_PA, abs_tol=1e-9)

assert all(
    right > left
    for left, right in zip(curve.slip_angle_rad, curve.slip_angle_rad[1:])
)
assert curve.slip_angle_rad[0] < 0.0 < curve.slip_angle_rad[-1]
assert left_slope > 0.0
assert right_slope > 0.0
assert curve.source_tire_id != curve.intended_tire_id

print("Lesson 02 tire-model checks passed.")
print("No extrapolation, symmetry repair, track scaling, or educational refit was introduced.")

## 9. Engineering takeaway

The useful relationship is now

[
\text{wheel heading + wheel-center motion}
\;\rightarrow\;
\alpha
\;\rightarrow\;
F_y.
]

The important distinction is between a **local model** and the **full measured response**:

[
F_y \approx F_y(0)+C_\alpha\alpha
]

is useful near zero because it gives a compact sensitivity. It is not the tire.

The reviewed runtime curve preserves the nonlinear transition, peak behavior, sign convention, source identity, and the possibility of multiple roots. That is exactly why the production model is a better teaching visual than a generic cartoon curve.

### Check yourself

1. If wheel heading and wheel-center velocity heading are identical, what is (alpha)?
2. Why can (F_y(0)) be nonzero in real processed tire data?
3. What does (C_\alpha) tell you physically?
4. Why does the tangent line eventually overpredict the real curve?
5. Why does a nonlinear inverse need a branch policy?

### Handoff to Lesson 03

Today we held (F_z), inclination, and pressure fixed.

Next question:

> **What happens to this entire (F_y(\alpha)) curve when vertical load or camber changes?**